# P4AI — Classical ML on Animals-10

Hand-crafted features + classical classifiers.

**Runtime:** Colab, T4 GPU

Five feature configurations, each evaluated with logistic regression, RBF-SVM and XGBoost
(random search on the validation split, model chosen by validation macro-F1):

| config | features |
|---|---|
| A1 HOG | HOG (8x8 cells, 2x2 blocks, L2-Hys) on 128x128 images |
| A2 HOG+LBP+HSV | HOG + uniform-LBP texture + HSV colour histograms |
| A3 BoVW+LBP+HSV | bag of visual words over HOG-block descriptors (GPU k-means, 1x1 + 2x2 pyramid) + LBP + HSV |
| A4 SIFT-BoVW | SIFT (DoG keypoints, up to 500 per image), RootSIFT, bag of visual words + spatial pyramid |
| A5 SIFT-BoVW+LBP+HSV+HOG | SIFT words + LBP + HSV + HOG, each block L2-normalized |

Split: stratified 70 / 15 / 15, seed 42

# 0. Setup

In [ ]:
import os, sys, json, time, math, subprocess, warnings, random, glob, gc, re, io, tempfile

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision.datasets import ImageFolder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")


def pip_install(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)


try:
    import cv2
    assert hasattr(cv2, "SIFT_create")
except (ImportError, AssertionError):
    pip_install("opencv-python-headless"); import cv2
try:
    import xgboost as xgb
except ImportError:
    pip_install("xgboost"); import xgboost as xgb

# cuML = GPU scikit-learn, used for the RBF-SVM (falls back to scikit-learn on CPU).
USE_CUML = False
if torch.cuda.is_available():
    try:
        import cuml
    except ImportError:
        print("cuML not found, installing (a few minutes)...")
        pip_install("--extra-index-url=https://pypi.nvidia.com", "cuml-cu12")
        try:
            import cuml
        except ImportError:
            cuml = None
    if cuml is not None:
        cuml.set_global_output_type("numpy")
        USE_CUML = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")
print("cuML available:", USE_CUML, "| OpenCV", cv2.__version__, "| XGBoost", xgb.__version__)

# 1. Configuration

In [ ]:
SEED = 42

IMG_LOAD = 224            # images are first resized to 224x224, like the DL notebook
IMG_SIZE_A = 128          # HOG / LBP / HSV work on a 128x128 downsample (GPU)
BATCH = 128
NUM_WORKERS = min(4, os.cpu_count() or 2)

BOVW_K = 512              # vocabulary size for HOG-block words (A3)
KMEANS_SAMPLE = 400_000   # descriptors sampled from TRAIN images to fit a codebook
KMEANS_ITERS = 30

SIFT_IMG = 224                 # SIFT runs on the 224x224 grayscale image
SIFT_MAX_KP = 500              # strongest DoG keypoints kept per image
SIFT_K = 512                   # vocabulary size for SIFT words
SIFT_CODEBOOK_IMAGES = 4000    # TRAIN images used to fit the SIFT codebook
SIFT_PER_IMAGE_SAMPLE = 100    # descriptors sampled per image for k-means

# random-search budget per (configuration, classifier)
N_TRIALS = {"logreg": 8, "svm_rbf": 8, "xgb": 4}
CLASSIFIERS = ["logreg", "svm_rbf", "xgb"]
if not USE_CUML:
    # scikit-learn's CPU SVC is ~10-50x slower on 18k samples: keep its budget small
    N_TRIALS["svm_rbf"] = min(N_TRIALS["svm_rbf"], 3)

VIZ_N_TEST = 6
VIZ_SEED = SEED + 1337
COST_N = 20                    # test images used to measure inference FLOPs (section 12.4)

TRANSLATE = {"cane": "dog", "cavallo": "horse", "elefante": "elephant",
             "farfalla": "butterfly", "gallina": "chicken", "gatto": "cat",
             "mucca": "cow", "pecora": "sheep", "ragno": "spider",
             "scoiattolo": "squirrel"}

torch.manual_seed(SEED); random.seed(SEED); np.random.seed(SEED)

# 2. Data and splits

In [ ]:
def get_animals10():
    """Download Animals-10 with kagglehub (cached) and return the path to raw-img/."""
    try:
        import kagglehub
    except ImportError:
        pip_install("kagglehub"); import kagglehub
    path = kagglehub.dataset_download("alessiocorrado99/animals10")
    hits = sorted(glob.glob(os.path.join(path, "**", "raw-img"), recursive=True))
    if not hits:
        raise RuntimeError(f"raw-img/ not found under {path}")
    return hits[0].rstrip("/") + "/"


DATA_DIR = get_animals10()
print("DATA_DIR:", DATA_DIR)

base = ImageFolder(DATA_DIR)
PATHS = [p for p, _ in base.samples]
LABELS = np.array([l for _, l in base.samples], dtype=np.int64)
CLASS_NAMES = [TRANSLATE.get(c, c) for c in base.classes]
N_CLASSES = len(CLASS_NAMES)


def make_splits(labels, train_ratio=0.70, val_ratio=0.15, seed=SEED):
    """Stratified 70/15/15 split, identical to the DL notebook."""
    indices = list(range(len(labels)))
    train_idx, temp_idx, _, y_temp = train_test_split(
        indices, list(labels), test_size=1 - train_ratio, stratify=list(labels), random_state=seed)
    rel_val = val_ratio / (1 - train_ratio)
    val_idx, test_idx, _, _ = train_test_split(
        temp_idx, y_temp, test_size=1 - rel_val, stratify=y_temp, random_state=seed)
    return np.array(train_idx), np.array(val_idx), np.array(test_idx)


tr, va, te = make_splits(LABELS)
SPLITS = {"train": tr, "val": va, "test": te}
TEST_IDX, Y_TEST = SPLITS["test"], LABELS[SPLITS["test"]]
print(f"{len(PATHS)} images, {N_CLASSES} classes {CLASS_NAMES}\n"
      f"train {len(tr)} / val {len(va)} / test {len(te)}")

# 3. GPU feature extractors (HOG, LBP, HSV)

In [ ]:
def hog_cells(x, cell=8, nbins=9):
    """Per-cell orientation histograms. x: (B,3,H,W) in [0,1] -> (B, nbins, H/cell, W/cell).
    Gradient = channel with the largest magnitude; unsigned orientation, 9 bins
    with linear interpolation between bins."""
    B, _, H, W = x.shape
    xp = F.pad(x, (1, 1, 1, 1), mode="replicate")
    gx = xp[:, :, 1:-1, 2:] - xp[:, :, 1:-1, :-2]
    gy = xp[:, :, 2:, 1:-1] - xp[:, :, :-2, 1:-1]
    mag = torch.sqrt(gx ** 2 + gy ** 2)
    best = mag.argmax(1, keepdim=True)
    gx, gy, mag = (t.gather(1, best).squeeze(1) for t in (gx, gy, mag))
    ang = torch.atan2(gy, gx) % math.pi
    pos = ang / (math.pi / nbins) - 0.5
    lo_f = torch.floor(pos)
    w_hi = pos - lo_f
    lo = lo_f.long() % nbins
    hi = (lo + 1) % nbins
    hist = torch.zeros(B, nbins, H, W, device=x.device, dtype=mag.dtype)
    hist.scatter_add_(1, lo.unsqueeze(1), (mag * (1 - w_hi)).unsqueeze(1))
    hist.scatter_add_(1, hi.unsqueeze(1), (mag * w_hi).unsqueeze(1))
    return F.avg_pool2d(hist, cell)                        # (B, nbins, Hc, Wc)


def hog_blocks_from_cells(cells, eps=1e-6):
    """2x2-cell blocks, stride 1, L2-Hys -> (B, n_blocks, 4*nbins)."""
    B, nbins = cells.shape[:2]
    blk = cells.unfold(2, 2, 1).unfold(3, 2, 1)            # (B, nbins, Hc-1, Wc-1, 2, 2)
    nby, nbx = blk.shape[2], blk.shape[3]
    blk = blk.permute(0, 2, 3, 1, 4, 5).reshape(B, nby * nbx, nbins * 4)
    blk = blk / (blk.norm(dim=-1, keepdim=True) + eps)
    blk = blk.clamp(max=0.2)
    return blk / (blk.norm(dim=-1, keepdim=True) + eps)


def hog_blocks(x, cell=8, nbins=9, eps=1e-6):
    """Dalal-Triggs HOG on a batch. x: (B,3,H,W) in [0,1] -> (B, n_blocks, 4*nbins)."""
    return hog_blocks_from_cells(hog_cells(x, cell, nbins), eps)


def grid_hist(codes, nbins, grids=(1, 2)):
    """Per-image histograms of integer codes over a spatial pyramid.
    codes: (B,H,W) long. Returns (B, sum(g*g)*nbins), L1-normalized per region."""
    B, H, W = codes.shape
    dev = codes.device
    out = []
    for g in grids:
        ry = torch.arange(H, device=dev) * g // H
        rx = torch.arange(W, device=dev) * g // W
        region = ry[:, None] * g + rx[None, :]
        flat = (torch.arange(B, device=dev)[:, None, None] * (g * g) + region[None]) * nbins + codes
        h = torch.bincount(flat.reshape(-1), minlength=B * g * g * nbins)
        h = h.view(B, g * g, nbins).float()
        h = h / h.sum(-1, keepdim=True).clamp(min=1)
        out.append(h.reshape(B, -1))
    return torch.cat(out, 1)


def _uniform_lbp_lut():
    lut = np.full(256, 58, dtype=np.int64)
    k = 0
    for c in range(256):
        bits = [(c >> i) & 1 for i in range(8)]
        if sum(bits[i] != bits[(i + 1) % 8] for i in range(8)) <= 2:
            lut[c] = k; k += 1
    assert k == 58
    return torch.from_numpy(lut)


LBP_LUT = _uniform_lbp_lut().to(DEVICE)


def rgb_to_gray(x):
    return 0.299 * x[:, 0] + 0.587 * x[:, 1] + 0.114 * x[:, 2]


def lbp_codes(gray):
    """Uniform LBP (P=8, R=1) code map, values 0..58. gray: (B,H,W)."""
    H, W = gray.shape[1:]
    p = F.pad(gray[:, None], (1, 1, 1, 1), mode="replicate")[:, 0]
    offs = [(-1, -1), (-1, 0), (-1, 1), (0, 1), (1, 1), (1, 0), (1, -1), (0, -1)]
    code = torch.zeros_like(gray, dtype=torch.long)
    for i, (dy, dx) in enumerate(offs):
        nb = p[:, 1 + dy:1 + dy + H, 1 + dx:1 + dx + W]
        code |= (nb >= gray).long() << i
    return LBP_LUT[code]


def lbp_hist(gray, grids=(1, 2)):
    """Uniform LBP histograms, 59 bins, over a 1x1 + 2x2 grid -> 295 dims."""
    return grid_hist(lbp_codes(gray), 59, grids)


def rgb_to_hsv(x):
    r, g, b = x[:, 0], x[:, 1], x[:, 2]
    maxc, _ = x.max(1); minc, _ = x.min(1)
    delta = maxc - minc
    d = delta + 1e-8
    h = torch.where(maxc == r, ((g - b) / d) % 6,
                    torch.where(maxc == g, (b - r) / d + 2, (r - g) / d + 4)) / 6
    h = torch.where(delta > 0, h, torch.zeros_like(h)) % 1.0
    s = delta / (maxc + 1e-8)
    return h, s, maxc


def hsv_hist(x, bins=(16, 4, 4), grids=(1, 2)):
    """Joint HSV histogram (16x4x4 = 256 bins) over a 1x1 + 2x2 grid -> 1280 dims."""
    h, s, v = rgb_to_hsv(x)
    hb = (h * bins[0]).long().clamp(0, bins[0] - 1)
    sb = (s * bins[1]).long().clamp(0, bins[1] - 1)
    vb = (v * bins[2]).long().clamp(0, bins[2] - 1)
    code = (hb * bins[1] + sb) * bins[2] + vb
    return grid_hist(code, bins[0] * bins[1] * bins[2], grids)

# 4. HOG / LBP / HSV extraction
One decoding pass over all images; the feature matrices are kept in memory (`FEATS`).

In [ ]:
FEATS = {}                                   # feature name -> (N, ...) numpy array
CODEBOOKS = {}                               # feature name -> (K, d) GPU tensor
META = {"extract_seconds": {}, "descriptors_per_image": {}}


class RawImages(Dataset):
    """Returns uint8 (3,S,S) images, resized like the DL notebook's Resize((S,S))."""
    def __init__(self, paths, size):
        self.paths, self.size = paths, size

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB").resize((self.size, self.size), Image.BILINEAR)
        return torch.from_numpy(np.asarray(img).copy()).permute(2, 0, 1), i


def _sync():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()


class managed_loader:
    """Iterate a multi-worker DataLoader safely in a notebook: collect stale iterators
    first, and always shut the workers down on exit (also when interrupted)."""
    def __init__(self, dl, desc):
        self.dl, self.desc = dl, desc

    def __enter__(self):
        gc.collect()
        self.it = iter(self.dl)
        return tqdm(self.it, total=len(self.dl), desc=self.desc)

    def __exit__(self, *exc):
        shutdown = getattr(self.it, "_shutdown_workers", None)
        if shutdown is not None:
            shutdown()
        self.it = None
        return False


def extract_raw():
    N = len(PATHS)
    gpu_s = 0.0
    dl = DataLoader(RawImages(PATHS, IMG_LOAD), batch_size=BATCH, shuffle=False,
                    num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda")
    t_start = time.time()
    with torch.no_grad(), managed_loader(dl, "HOG/LBP/HSV") as batches:
        for xb, ib in batches:
            sl = slice(int(ib[0]), int(ib[-1]) + 1)
            x = xb.to(DEVICE, non_blocking=True).float().div_(255)
            _sync(); t0 = time.time()
            xa = F.interpolate(x, size=(IMG_SIZE_A, IMG_SIZE_A), mode="bilinear",
                               antialias=True, align_corners=False).clamp(0, 1)
            feats = {"hog": hog_blocks(xa), "lbp": lbp_hist(rgb_to_gray(xa)), "hsv": hsv_hist(xa)}
            _sync(); gpu_s += time.time() - t0
            for n, f in feats.items():
                if n not in FEATS:
                    dtype = np.float16 if n == "hog" else np.float32      # HOG blocks stored as fp16
                    FEATS[n] = np.zeros((N,) + tuple(f.shape[1:]), dtype=dtype)
                FEATS[n][sl] = f.float().cpu().numpy()
    META["extract_seconds"]["trackA"] = round(gpu_s, 1)
    META["extract_seconds"]["trackA_wall"] = round(time.time() - t_start, 1)
    print({n: a.shape for n, a in FEATS.items()})


extract_raw()

# 5. GPU k-means and HOG-block bag of visual words (A3)
Codebook fitted on descriptors from TRAIN images only; each image becomes a 1x1 + 2x2
spatial-pyramid histogram of visual words, Hellinger (sqrt) + L2 normalized.

In [ ]:
def nearest_center(X, C, chunk=131072):
    cn = (C * C).sum(1)
    out = torch.empty(X.shape[0], dtype=torch.long, device=X.device)
    for s in range(0, X.shape[0], chunk):
        out[s:s + chunk] = (cn[None, :] - 2 * X[s:s + chunk] @ C.T).argmin(1)
    return out


def kmeans(X, K, iters=30, seed=SEED):
    g = torch.Generator(device=X.device).manual_seed(seed)
    C = X[torch.randperm(X.shape[0], generator=g, device=X.device)[:K]].clone()
    for _ in range(iters):
        a = nearest_center(X, C)
        cnt = torch.bincount(a, minlength=K).float()
        S = torch.zeros_like(C).index_add_(0, a, X)
        empty = cnt == 0
        C = torch.where(empty[:, None], C, S / cnt.clamp(min=1)[:, None])
        if empty.any():   # re-seed dead centres with random descriptors
            ridx = torch.randint(X.shape[0], (int(empty.sum()),), generator=g, device=X.device)
            C[empty] = X[ridx]
    return C


HOG_BOVW = f"bovw_K{BOVW_K}"


def build_hog_bovw(K=BOVW_K):
    t0 = time.time()
    blocks = FEATS["hog"]                                       # (N, nb, d) fp16
    N, nb, d = blocks.shape
    side = int(round(math.sqrt(nb)))
    train_desc = torch.from_numpy(blocks[np.sort(SPLITS["train"])]).reshape(-1, d)
    g = torch.Generator().manual_seed(SEED)
    pick = torch.randperm(train_desc.shape[0], generator=g)[:KMEANS_SAMPLE]
    sample = train_desc[pick].to(DEVICE).float()
    del train_desc
    C = kmeans(sample, K, KMEANS_ITERS)
    CODEBOOKS[HOG_BOVW] = C
    enc = np.zeros((N, 5 * K), dtype=np.float16)
    for s in tqdm(range(0, N, 1024), desc="encode HOG-BoVW"):
        b = torch.from_numpy(blocks[s:s + 1024]).to(DEVICE).float()
        B = b.shape[0]
        words = nearest_center(b.reshape(-1, d), C).view(B, side, side)
        h = grid_hist(words, K, grids=(1, 2)).sqrt()
        enc[s:s + B] = (h / h.norm(dim=1, keepdim=True).clamp(min=1e-12)).cpu().numpy()
    FEATS[HOG_BOVW] = enc
    META["extract_seconds"][HOG_BOVW] = round(time.time() - t0, 1)
    print(f"HOG-BoVW K={K}: {enc.shape} in {time.time() - t0:.1f}s")


build_hog_bovw()

# 6. SIFT bag of visual words (A4, A5)
SIFT descriptors are computed with OpenCV in the DataLoader workers (CPU), as RootSIFT.
Two phases: (1) descriptors of `SIFT_CODEBOOK_IMAGES` random TRAIN images -> GPU k-means
codebook; (2) one pass over all images, each encoded at once as a 1x1 + 2x2 spatial-pyramid
word histogram (from keypoint positions), Hellinger + L2 normalized.

In [ ]:
SIFT_NAME = f"sift_kp_K{SIFT_K}"


def root_sift(d):
    d = d / (np.abs(d).sum(1, keepdims=True) + 1e-7)
    return np.sqrt(d).astype(np.float32)


class SiftImages(Dataset):
    """Grayscale 224x224 image -> RootSIFT descriptors and (x, y) keypoint positions.
    The OpenCV object is created lazily inside each worker process."""
    def __init__(self, paths, idxs):
        self.paths, self.idxs = paths, np.asarray(idxs)
        self._sift = None

    def __len__(self):
        return len(self.idxs)

    def __getitem__(self, j):
        if self._sift is None:
            cv2.setNumThreads(1)
            self._sift = cv2.SIFT_create(nfeatures=SIFT_MAX_KP)
        i = int(self.idxs[j])
        g = np.asarray(Image.open(self.paths[i]).convert("L").resize((SIFT_IMG, SIFT_IMG), Image.BILINEAR))
        kps, d = self._sift.detectAndCompute(g, None)
        if d is None or len(kps) == 0:
            return {"i": i, "kp": (np.zeros((0, 128), np.float32), np.zeros((0, 2), np.float32))}
        return {"i": i, "kp": (root_sift(d), np.array([k.pt for k in kps], np.float32))}


def list_collate(batch):
    return batch


def encode_sift_batch(items, C, K):
    """Spatial-pyramid BoVW (1x1 + 2x2) for a batch of images -> (B, 5K)."""
    B = len(items)
    descs, ids, regs = [], [], []
    for b, it in enumerate(items):
        d, xy = it["kp"]
        if len(d) == 0:
            continue
        descs.append(d)
        ids.append(np.full(len(d), b))
        rx = np.clip((xy[:, 0] * 2 // SIFT_IMG).astype(np.int64), 0, 1)
        ry = np.clip((xy[:, 1] * 2 // SIFT_IMG).astype(np.int64), 0, 1)
        regs.append(ry * 2 + rx)
    if not descs:
        return torch.zeros(B, 5 * K)
    D = torch.from_numpy(np.concatenate(descs)).to(DEVICE)
    img = torch.from_numpy(np.concatenate(ids)).to(DEVICE)
    reg = torch.from_numpy(np.concatenate(regs)).to(DEVICE)
    w = nearest_center(D, C)
    h0 = torch.bincount(img * K + w, minlength=B * K).view(B, 1, K)
    h1 = torch.bincount((img * 4 + reg) * K + w, minlength=B * 4 * K).view(B, 4, K)
    h = torch.cat([h0, h1], 1).float()
    h = (h / h.sum(-1, keepdim=True).clamp(min=1)).sqrt().reshape(B, -1)
    return (h / h.norm(dim=1, keepdim=True).clamp(min=1e-12)).cpu()


def build_sift_bovw(K=SIFT_K):
    # phase 1: codebook from a sample of TRAIN images
    t0 = time.time()
    rng = np.random.default_rng(SEED)
    sub = rng.choice(SPLITS["train"], size=min(SIFT_CODEBOOK_IMAGES, len(SPLITS["train"])), replace=False)
    dl = DataLoader(SiftImages(PATHS, sub), batch_size=32, num_workers=NUM_WORKERS, collate_fn=list_collate)
    samples = []
    with managed_loader(dl, "SIFT codebook sample") as batches:
        for batch in batches:
            for it in batch:
                d = it["kp"][0]
                if len(d) > SIFT_PER_IMAGE_SAMPLE:
                    d = d[rng.choice(len(d), SIFT_PER_IMAGE_SAMPLE, replace=False)]
                samples.append(d)
    X = torch.from_numpy(np.concatenate(samples)).to(DEVICE)
    C = kmeans(X, K, KMEANS_ITERS)
    CODEBOOKS[SIFT_NAME] = C
    print(f"  codebook: {X.shape[0]} descriptors -> {K} words")
    del samples, X
    t_codebook = time.time() - t0

    # phase 2: encode every image
    t1 = time.time()
    N = len(PATHS)
    enc = np.zeros((N, 5 * K), np.float16)
    n_desc = np.zeros(N, np.int32)
    dl = DataLoader(SiftImages(PATHS, np.arange(N)), batch_size=64, num_workers=NUM_WORKERS,
                    collate_fn=list_collate)
    with torch.no_grad(), managed_loader(dl, "SIFT encode") as batches:
        for batch in batches:
            ids = np.array([it["i"] for it in batch])
            enc[ids] = encode_sift_batch(batch, C, K).numpy()
            n_desc[ids] = [len(it["kp"][0]) for it in batch]
    FEATS[SIFT_NAME] = enc
    META["extract_seconds"][SIFT_NAME] = round(t_codebook + time.time() - t1, 1)
    META["descriptors_per_image"][SIFT_NAME] = {
        "mean": float(n_desc.mean()), "min": int(n_desc.min()), "max": int(n_desc.max()),
        "images_with_zero": int((n_desc == 0).sum())}
    print(f"SIFT BoVW {enc.shape}: codebook {t_codebook:.0f}s + encode {time.time() - t1:.0f}s | "
          f"descriptors/image {META['descriptors_per_image'][SIFT_NAME]}")


build_sift_bovw()
print(json.dumps(META["extract_seconds"], indent=2))

# 7. Feature assembly and preprocessing (fitted on TRAIN only)

In [ ]:
_block_cache = {}


def load_block(name):
    if name not in _block_cache:
        arr = FEATS[name].astype(np.float32)
        _block_cache[name] = arr.reshape(arr.shape[0], -1)
    return _block_cache[name]


def l2n(X):
    return X / X.norm(dim=1, keepdim=True).clamp(min=1e-12)


def normalize_block(X, how):
    if how is None:
        return X
    if "sqrt" in how:
        X = X.clamp(min=0).sqrt()      # Hellinger map for histograms
    if "l2" in how:
        X = l2n(X)
    return X


def assemble(blocks):
    """blocks: list of (feature_name, normalization). Each block is normalized on its own
    before concatenation so no block dominates by scale."""
    return torch.cat([normalize_block(torch.from_numpy(load_block(n)).to(DEVICE), how)
                      for n, how in blocks], dim=1)


class Prep:
    """Sequential preprocessing; every statistic is computed on TRAIN rows only.
    Steps: ("std",)  ("l2",)  ("pca", dim)  ("pca", dim, True) for whitening."""
    def __init__(self, steps):
        self.steps = steps
        self.info = {}
        self.params = []

    def fit_transform(self, X, tr):
        n = len(tr)
        self.params = []
        for step in self.steps:
            kind = step[0]
            if kind == "l2":
                p = None
            elif kind == "std":
                p = {"mu": X[tr].mean(0), "sd": X[tr].std(0).clamp(min=1e-6)}
            elif kind == "pca":
                dim = min(step[1], X.shape[1], n - 1)
                whiten = len(step) > 2 and step[2]
                mu = X[tr].mean(0)
                Xc = X[tr] - mu
                total_var = (Xc ** 2).sum() / (n - 1)
                _, S, V = torch.pca_lowrank(Xc, q=dim, center=False, niter=4)
                ev = S[:dim] ** 2 / (n - 1)
                self.info["pca_explained_var"] = float(ev.sum() / total_var)
                p = {"mu": mu, "V": V[:, :dim].contiguous(),
                     "scale": ev.sqrt().clamp(min=1e-6) if whiten else None}
                del Xc
            else:
                raise ValueError(kind)
            self.params.append(p)
            X = self._apply(kind, p, X)
        return X

    @staticmethod
    def _apply(kind, p, X):
        if kind == "l2":
            return l2n(X)
        if kind == "std":
            return (X - p["mu"]) / p["sd"]
        X = (X - p["mu"]) @ p["V"]
        return X / p["scale"] if p["scale"] is not None else X

    def transform(self, X):
        for step, p in zip(self.steps, self.params):
            X = self._apply(step[0], p, X)
        return X

    def n_params(self):
        return sum(v.numel() for p in self.params if p for v in p.values() if torch.is_tensor(v))


class FeatSet:
    """One preprocessed feature matrix split into train/val/test, as GPU tensors
    and (lazily) as numpy arrays."""
    def __init__(self, X, labels, splits):
        y = torch.from_numpy(labels).to(DEVICE)
        self.Xt = {k: X[torch.from_numpy(v).to(DEVICE)].contiguous() for k, v in splits.items()}
        self.yt = {k: y[torch.from_numpy(v).to(DEVICE)] for k, v in splits.items()}
        self._np = {}

    def Xn(self, k):
        if k not in self._np:
            self._np[k] = np.ascontiguousarray(self.Xt[k].float().cpu().numpy())
        return self._np[k]

    def yn(self, k):
        return self.yt[k].cpu().numpy()

    @property
    def dim(self):
        return self.Xt["train"].shape[1]

# 8. Classifiers and random-search tuning
All fits use TRAIN; hyperparameters are chosen by macro-F1 on VAL; TEST is scored once.

In [ ]:
def macro_f1(y, p):
    return precision_recall_fscore_support(y, p, average="macro", zero_division=0)[2]


def loguniform(rng, lo, hi):
    return float(10 ** rng.uniform(math.log10(lo), math.log10(hi)))


def to_np(X):
    if torch.is_tensor(X):
        return np.ascontiguousarray(X.float().cpu().numpy())
    return np.ascontiguousarray(X, dtype=np.float32)


class LogReg:
    """Multinomial logistic regression, L2-penalized, full-batch L-BFGS on GPU."""
    @staticmethod
    def sample(rng, fs):
        return {"lam": loguniform(rng, 1e-6, 1e-1)}

    def __init__(self, p):
        self.lam = p["lam"]

    def fit(self, fs):
        X, y = fs.Xt["train"].float(), fs.yt["train"]
        self.W = torch.zeros(X.shape[1], N_CLASSES, device=X.device, requires_grad=True)
        self.b = torch.zeros(N_CLASSES, device=X.device, requires_grad=True)
        opt = torch.optim.LBFGS([self.W, self.b], lr=1, max_iter=300, history_size=20,
                                line_search_fn="strong_wolfe")

        def closure():
            opt.zero_grad()
            loss = F.cross_entropy(X @ self.W + self.b, y) + self.lam * (self.W ** 2).sum()
            loss.backward()
            return loss
        opt.step(closure)
        self.W, self.b = self.W.detach(), self.b.detach()
        return self

    def predict_X(self, X):
        X = X if torch.is_tensor(X) else torch.from_numpy(to_np(X))
        with torch.no_grad():
            return (X.to(DEVICE).float() @ self.W + self.b).argmax(1).cpu().numpy()

    def predict(self, fs, split):
        return self.predict_X(fs.Xt[split])

    def scores(self, X):
        """(n, n_classes) class scores and their kind: softmax probabilities."""
        X = X if torch.is_tensor(X) else torch.from_numpy(to_np(X))
        with torch.no_grad():
            return torch.softmax(X.to(DEVICE).float() @ self.W + self.b, 1).cpu().numpy(), "prob"

    def save(self, path):
        torch.save({"lam": self.lam, "W": self.W.cpu(), "b": self.b.cpu()}, path)


class SVMRBF:
    """RBF-kernel SVM (cuML on GPU, scikit-learn fallback).
    gamma = g / (D * var(X_train)), i.e. g scales scikit-learn's 'scale' default."""
    @staticmethod
    def sample(rng, fs):
        return {"C": loguniform(rng, 0.1, 100), "g": loguniform(rng, 0.1, 10)}

    def __init__(self, p):
        self.p = p

    def fit(self, fs):
        Xtr = fs.Xn("train")
        gamma = self.p["g"] / (Xtr.shape[1] * Xtr.var())
        if USE_CUML:
            from cuml.svm import SVC
        else:
            from sklearn.svm import SVC
        self.m = SVC(kernel="rbf", C=self.p["C"], gamma=gamma, cache_size=2048)
        self.m.fit(Xtr, fs.yn("train").astype(np.int32))
        return self

    def predict_X(self, X):
        return np.asarray(self.m.predict(to_np(X))).astype(np.int64)

    def predict(self, fs, split):
        return self.predict_X(fs.Xn(split))

    def scores(self, X):
        """Probabilities if the SVM has them, else one-vs-rest decision values ("margin"),
        else one-hot predictions ("none")."""
        Xn = to_np(X)
        for fn, kind in (("predict_proba", "prob"), ("decision_function", "margin")):
            f = getattr(self.m, fn, None)
            if f is None:
                continue
            try:
                sc = np.asarray(f(Xn), dtype=np.float32)
                if sc.ndim == 2 and sc.shape[1] == N_CLASSES:
                    return sc, kind
            except Exception:
                pass
        return np.eye(N_CLASSES, dtype=np.float32)[self.predict_X(Xn)], "none"

    def save(self, path):
        import pickle
        with open(path, "wb") as f:
            pickle.dump({"p": self.p, "m": self.m}, f)


class XGB:
    """Gradient-boosted trees on GPU; early stopping on VAL log-loss."""
    @staticmethod
    def sample(rng, fs):
        return {"max_depth": int(rng.choice([4, 6, 8])),
                "learning_rate": loguniform(rng, 0.05, 0.3),
                "subsample": float(rng.uniform(0.6, 1.0)),
                "colsample_bytree": float(rng.uniform(0.3, 0.9)),
                "min_child_weight": int(rng.choice([1, 3, 5])),
                "reg_lambda": loguniform(rng, 0.1, 10)}

    def __init__(self, p):
        self.p = p

    def fit(self, fs):
        self.m = xgb.XGBClassifier(
            n_estimators=1500, early_stopping_rounds=50, tree_method="hist",
            device="cuda" if DEVICE.type == "cuda" else "cpu",
            eval_metric="mlogloss", random_state=SEED, **self.p)
        self.m.fit(fs.Xn("train"), fs.yn("train"), eval_set=[(fs.Xn("val"), fs.yn("val"))], verbose=False)
        self.p_used = {**self.p, "best_iteration": int(self.m.best_iteration)}
        return self

    def predict_X(self, X):
        return self.m.predict(to_np(X)).astype(np.int64)

    def predict(self, fs, split):
        return self.predict_X(fs.Xn(split))

    def scores(self, X):
        return self.m.predict_proba(to_np(X)).astype(np.float32), "prob"

    def save(self, path):
        self.m.save_model(path)


CLF = {"logreg": LogReg, "svm_rbf": SVMRBF, "xgb": XGB}
CLF_EXT = {"logreg": "pt", "svm_rbf": "pkl", "xgb": "json"}


def search(clf_name, fs, rng, cfg_name):
    """Random search: N_TRIALS[clf_name] draws, the best by validation macro-F1 is kept."""
    cls = CLF[clf_name]
    grid = [cls.sample(rng, fs) for _ in range(N_TRIALS[clf_name])]
    best, trials = None, []
    for p in grid:
        t0 = time.time()
        m = cls(p).fit(fs)
        fit_s = time.time() - t0
        f1 = macro_f1(fs.yn("val"), m.predict(fs, "val"))
        used = getattr(m, "p_used", p)
        trials.append({"config": cfg_name, "classifier": clf_name,
                       "params": json.dumps(used), "val_f1": f1, "fit_s": fit_s})
        if best is None or f1 > best[2]:
            best = (m, used, f1, fit_s)
        else:
            del m
    return best, trials

# 9. Track A experiments
Every run (configuration x classifier) keeps its result row, search trials, test predictions,
best model and fitted preprocessing in memory (`RESULTS`, `TRIALS`, `PREDS`, `MODELS`, `PREPS`).

In [ ]:
TRACK_A = {
    "A1_HOG": {
        "blocks": [("hog", None)],                         # 225 blocks x 36 = 8100-d
        "prep": [("pca", 512), ("std",)]},
    "A2_HOG+LBP+HSV": {
        "blocks": [("hog", "l2"), ("lbp", "sqrt,l2"), ("hsv", "sqrt,l2")],
        "prep": [("pca", 512), ("std",)]},
    "A3_BoVW+LBP+HSV": {
        "blocks": [(HOG_BOVW, None), ("lbp", "sqrt,l2"), ("hsv", "sqrt,l2")],
        "prep": [("pca", 512), ("std",)]},
    "A4_SIFT-BoVW": {
        "blocks": [(SIFT_NAME, None)],                     # 5 x 512 = 2560-d
        "prep": [("pca", 512), ("std",)]},
    "A5_SIFT-BoVW+LBP+HSV+HOG": {                          # 2560+295+1280+8100-d
        "blocks": [(SIFT_NAME, None), ("lbp", "sqrt,l2"), ("hsv", "sqrt,l2"), ("hog", "l2")],
        "prep": [("pca", 512), ("std",)]},
}

RESULTS, TRIALS = [], []
PREDS, MODELS, PREPS = {}, {}, {}           # keyed "cfg|clf" (PREPS by cfg)


def disp(cfg_name):
    """Display name: the configuration key without its "A1_".."A5_" prefix."""
    return re.sub(r"^A\d+_", "", cfg_name)


def run_config(cfg_name, cfg):
    t0 = time.time()
    prep = Prep(cfg["prep"])
    X = prep.fit_transform(assemble(cfg["blocks"]), torch.from_numpy(SPLITS["train"]).to(DEVICE))
    PREPS[cfg_name] = prep
    fs = FeatSet(X, LABELS, SPLITS)
    del X
    print(f"\n=== {disp(cfg_name)}  dim={fs.dim}  prep={time.time() - t0:.1f}s  {prep.info}")
    rng = np.random.default_rng(SEED)
    for c in CLASSIFIERS:
        t1 = time.time()
        (m, params, val_f1, fit_s), trials = search(c, fs, rng, cfg_name)
        search_s = time.time() - t1
        t2 = time.time()
        pred = m.predict(fs, "test")
        pred_s = time.time() - t2
        y = fs.yn("test")
        p_, r_, f_, _ = precision_recall_fscore_support(y, pred, average="macro", zero_division=0)
        f1_cls = precision_recall_fscore_support(y, pred, average=None, zero_division=0,
                                                 labels=list(range(N_CLASSES)))[2]
        RESULTS.append({"config": cfg_name, "classifier": c, "dim": fs.dim,
                        "pca_explained_var": prep.info.get("pca_explained_var"),
                        "best_params": json.dumps(params), "val_f1": val_f1,
                        "test_acc": accuracy_score(y, pred), "test_prec": p_, "test_rec": r_,
                        "test_f1": f_, "fit_s_best": fit_s, "search_s": search_s,
                        "test_ms_per_img": 1000 * pred_s / len(y)})
        TRIALS.extend(trials)
        PREDS[f"{cfg_name}|{c}"] = {"y_true": y, "y_pred": pred, "f1_per_class": f1_cls}
        MODELS[f"{cfg_name}|{c}"] = m
        print(f"  {c:8s} val F1 {val_f1:.4f} | test acc {RESULTS[-1]['test_acc']:.4f} "
              f"F1 {f_:.4f} | search {search_s:.0f}s")
    del fs
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()


for name, cfg in TRACK_A.items():
    run_config(name, cfg)
_block_cache.clear()
_ = gc.collect()

# 10. Results

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.collections import LineCollection
from IPython.display import display

plt.rcParams.update({"savefig.dpi": 200, "figure.dpi": 110, "font.size": 9, "axes.titlesize": 10,
                     "axes.titleweight": "bold", "axes.spines.top": False,
                     "axes.spines.right": False, "legend.frameon": False})


def show_figure(fig):
    if fig is not None:
        display(fig); plt.close(fig)


def show_table(df):
    with pd.option_context("display.float_format", "{:.4f}".format, "display.width", 250,
                           "display.max_columns", 60):
        display(df)
    return df


def macro_f1_fast(y, p, k=10):
    ok = p >= 0
    cm = np.bincount(y[ok] * k + p[ok], minlength=k * k).reshape(k, k)
    tp = np.diag(cm)
    fp, fn = cm.sum(0) - tp, np.bincount(y, minlength=k) - tp
    return float((2 * tp / np.maximum(2 * tp + fp + fn, 1)).mean())


def bootstrap_ci(y, p, n=1000, seed=42):
    """95% percentile bootstrap CI of the test macro-F1 (1000 resamples of the test set)."""
    y, p = np.asarray(y), np.asarray(p)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(y), (n, len(y)))
    s = np.array([macro_f1_fast(y[i], p[i]) for i in idx])
    return np.percentile(s, [2.5, 97.5])


def confusion_panel(ax, y, p, class_names, title):
    """Row-normalized confusion matrix."""
    k = len(class_names)
    ok = p >= 0
    cm = np.bincount(y[ok] * k + p[ok], minlength=k * k).reshape(k, k)
    cm = cm / np.maximum(np.bincount(y, minlength=k)[:, None], 1)
    ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
    for i in range(k):
        for j in range(k):
            if cm[i, j] >= 0.01:
                ax.text(j, i, f"{cm[i, j]:.2f}", ha="center", va="center", fontsize=6,
                        color="white" if cm[i, j] > 0.5 else "black")
    ax.set_xticks(range(k)); ax.set_xticklabels(class_names, rotation=45, ha="right", fontsize=7)
    ax.set_yticks(range(k)); ax.set_yticklabels(class_names, fontsize=7)
    ax.set_xlabel("predicted", fontsize=8); ax.set_ylabel("true", fontsize=8)
    ax.set_title(title, fontsize=9)


def heatmap_panel(ax, values, row_labels, col_labels, cmap="RdYlGn", vmin=None, vmax=100, fmt="{:.0f}"):
    vmin = max(0, np.nanmin(values) - 5) if vmin is None else vmin
    im = ax.imshow(values, cmap=cmap, vmin=vmin, vmax=vmax, aspect="auto")
    for i in range(values.shape[0]):
        for j in range(values.shape[1]):
            ax.text(j, i, fmt.format(values[i, j]), ha="center", va="center", fontsize=7)
    ax.set_xticks(range(len(col_labels))); ax.set_xticklabels(col_labels, rotation=30, ha="right")
    ax.set_yticks(range(len(row_labels))); ax.set_yticklabels(row_labels, fontsize=8)
    return im


CLF_COLORS = {"logreg": "#4C72B0", "svm_rbf": "#DD8452", "xgb": "#55A868"}
CLF_SHORT = {"logreg": "LR", "svm_rbf": "SVM", "xgb": "XGB"}

res = pd.DataFrame(RESULTS)
res["pipeline"] = res["config"].map(disp)
RUNS = [(c, k) for c in TRACK_A for k in CLASSIFIERS if f"{c}|{k}" in PREDS]
BEST = res.loc[res.groupby("config")["val_f1"].idxmax()].set_index("config")   # best classifier per config (val)
print(f"{len(RUNS)} finished runs")

## 10.1 Results tables
All 15 runs: metrics, 95% bootstrap CI of the test macro-F1, cost; then test macro-F1 (%) as a
configuration x classifier pivot.

In [ ]:
def build_results_table():
    t = res.copy()
    ci = {(r.config, r.classifier): bootstrap_ci(Y_TEST, PREDS[f"{r.config}|{r.classifier}"]["y_pred"])
          for r in t.itertuples()}
    t["f1_ci_low"] = [ci[(c, k)][0] for c, k in zip(t["config"], t["classifier"])]
    t["f1_ci_high"] = [ci[(c, k)][1] for c, k in zip(t["config"], t["classifier"])]
    t["best_of_config"] = [BEST.loc[c, "classifier"] == k for c, k in zip(t["config"], t["classifier"])]
    cols = ["pipeline", "config", "classifier", "best_of_config", "dim", "pca_explained_var", "val_f1",
            "test_acc", "test_prec", "test_rec", "test_f1", "f1_ci_low", "f1_ci_high", "fit_s_best",
            "search_s", "test_ms_per_img", "best_params"]
    return t[cols].sort_values("test_f1", ascending=False).reset_index(drop=True)


TAB = show_table(build_results_table())
PIVOT = TAB.pivot_table(index="pipeline", columns="classifier", values="test_f1") * 100
PIVOT = show_table(PIVOT.reindex([disp(c) for c in TRACK_A if disp(c) in PIVOT.index]).round(2).reset_index())

## 10.2 Test macro-F1 by configuration and classifier (95% bootstrap CIs)

In [ ]:
def fig_f1_by_config():
    cfgs = [disp(c) for c in TRACK_A if disp(c) in set(TAB["pipeline"])]
    fig, ax = plt.subplots(figsize=(12, 4.8))
    w = 0.8 / len(CLASSIFIERS)
    for j, clf in enumerate(CLASSIFIERS):
        t = TAB[TAB["classifier"] == clf].set_index("pipeline").reindex(cfgs)
        x = np.arange(len(cfgs)) - 0.4 + w * (j + 0.5)
        f = t["test_f1"].values * 100
        err = [f - t["f1_ci_low"].values * 100, t["f1_ci_high"].values * 100 - f]
        bars = ax.bar(x, f, w, yerr=err, capsize=2, color=CLF_COLORS[clf], label=clf)
        ax.bar_label(bars, fmt="%.1f", fontsize=7, padding=3)
        star = t["best_of_config"].fillna(False).astype(bool).values
        ax.scatter(x[star], np.full(star.sum(), 3), marker="*", color="white", s=40, zorder=3)
    ax.set_xticks(range(len(cfgs))); ax.set_xticklabels(cfgs, rotation=10)
    ax.set_ylabel("test macro-F1 (%)"); ax.set_ylim(0, 105); ax.legend(ncol=3, loc="upper left")
    ax.set_title("Classical pipelines: test macro-F1 (error bar = 95% bootstrap CI, ★ = chosen on validation)")
    fig.tight_layout()
    return fig


show_figure(fig_f1_by_config())

## 10.3 Feature extraction cost

In [ ]:
def build_cost():
    rows = [{"step": k, "seconds": v} for k, v in META["extract_seconds"].items()]
    for name, d in META["descriptors_per_image"].items():
        rows.append({"step": f"{name}: SIFT descriptors / image", "seconds": np.nan,
                     **{f"desc_{k}": v for k, v in d.items()}})
    return pd.DataFrame(rows)


_ = show_table(build_cost())

## 10.4 Confusion matrices (all 15 runs) and per-class F1

In [ ]:
def fig_confusions():
    fig, axes = plt.subplots(len(TRACK_A), len(CLASSIFIERS), figsize=(3.4 * len(CLASSIFIERS), 3.3 * len(TRACK_A)),
                             squeeze=False)
    f1 = res.set_index(["config", "classifier"])["test_f1"]
    for i, c in enumerate(TRACK_A):
        for j, k in enumerate(CLASSIFIERS):
            ax = axes[i, j]
            if f"{c}|{k}" not in PREDS:
                ax.axis("off"); continue
            confusion_panel(ax, PREDS[f"{c}|{k}"]["y_true"], PREDS[f"{c}|{k}"]["y_pred"], CLASS_NAMES,
                            f"{disp(c)} / {k}  (F1 {f1.loc[(c, k)] * 100:.1f}%)")
    fig.tight_layout()
    return fig


def fig_per_class():
    keys = [f"{c}|{k}" for c, k in RUNS]
    vals = np.stack([PREDS[x]["f1_per_class"] * 100 for x in keys])
    fig, ax = plt.subplots(figsize=(11, 0.4 * len(keys) + 1.6))
    im = heatmap_panel(ax, vals, [f"{disp(x.split('|')[0])} / {x.split('|')[1]}" for x in keys], CLASS_NAMES)
    fig.colorbar(im, ax=ax, fraction=0.03, label="F1 (%)")
    ax.set_title("Per-class test F1 (%) of every classical run")
    fig.tight_layout()
    return fig


show_figure(fig_confusions())
show_figure(fig_per_class())

# 11. Features and predictions on test images
`VIZ_N_TEST` random test images (one per class, seed `VIZ_SEED`; the same images as the DL
notebook) + test.png, a deliberately confusing cat-butterfly hybrid with no label.

* **Features**: every Track A feature once (HOG, LBP, HSV, HOG words, SIFT words), one row per image.
* **Predictions**: the class distribution of the best classifier of each configuration
  (chosen on validation F1), log scale, true class shaded grey.

Features are recomputed from the image files exactly as in extraction and checked against the
extracted matrices; predictions are checked against the saved test predictions.

In [ ]:
def pick_test_images(test_idx, y_test, n_classes, n, seed):
    """n random test images, one per class while n <= n_classes, never the same image twice."""
    rng = np.random.default_rng(seed)
    if n <= n_classes:
        classes = rng.choice(n_classes, n, replace=False)
    else:
        classes = np.concatenate([rng.permutation(n_classes) for _ in range(math.ceil(n / n_classes))])[:n]
    out = []
    for c in classes:
        pool = test_idx[y_test == c]
        pool = pool[~np.isin(pool, out)]
        out.append(int(rng.choice(pool)))
    return np.array(out, dtype=np.int64)


DEMO_IDX = pick_test_images(TEST_IDX, Y_TEST, N_CLASSES, VIZ_N_TEST, VIZ_SEED)
CONFUSE_ID = "19UAsXlHKbs_wEB-vw2Px0ZkzIQhq13Uj"          # public file id of test.png
CONFUSE_PATH = os.path.abspath("./test.png")


def is_image(path):
    try:
        with Image.open(path) as im:
            im.verify()
        return True
    except Exception:
        return False


def get_confusing_image():
    if not (os.path.exists(CONFUSE_PATH) and is_image(CONFUSE_PATH)):
        try:
            import gdown
        except ImportError:
            pip_install("gdown"); import gdown
        gdown.download(id=CONFUSE_ID, output=CONFUSE_PATH)
    if os.path.exists(CONFUSE_PATH) and is_image(CONFUSE_PATH):
        return CONFUSE_PATH
    if os.path.exists(CONFUSE_PATH):
        os.remove(CONFUSE_PATH)
    return None


def dataset_item(i):
    return {"idx": int(i), "path": PATHS[i], "true": int(LABELS[i]), "label": f"#{i} {CLASS_NAMES[LABELS[i]]}"}


DEMO_ITEMS = [dataset_item(i) for i in DEMO_IDX]
if get_confusing_image():
    DEMO_ITEMS.append({"idx": -1, "path": CONFUSE_PATH, "true": -1, "label": "test.png: cat-butterfly hybrid"})
else:
    print("WARNING: test.png could not be downloaded; upload it to /content and rerun this cell.")
print("Images:", [it["label"] for it in DEMO_ITEMS])

In [ ]:
C_HOG = CODEBOOKS[HOG_BOVW].float()
C_SIFT = CODEBOOKS[SIFT_NAME].float()
_SIFT_MAIN = cv2.SIFT_create(nfeatures=SIFT_MAX_KP)


def sift_keypoints(gray_u8):
    kps, d = _SIFT_MAIN.detectAndCompute(gray_u8, None)
    if d is None or len(kps) == 0:
        return [], np.zeros((0, 128), np.float32), np.zeros((0, 2), np.float32)
    return kps, root_sift(d), np.array([k.pt for k in kps], np.float32)


def compute_features(imgs, grays):
    """Recompute every Track A feature for uint8 images (B,224,224,3), exactly as in
    extraction (including fp16 storage rounding), plus what the plots need."""
    x = torch.from_numpy(np.ascontiguousarray(imgs)).to(DEVICE).permute(0, 3, 1, 2).float() / 255
    B = x.shape[0]
    xa = F.interpolate(x, size=(IMG_SIZE_A, IMG_SIZE_A), mode="bilinear", antialias=True,
                       align_corners=False).clamp(0, 1)
    cells = hog_cells(xa)
    blocks = hog_blocks_from_cells(cells).half().float()          # stored as fp16
    codes = lbp_codes(rgb_to_gray(xa))
    feats = {"hog": blocks.reshape(B, -1), "lbp": grid_hist(codes, 59), "hsv": hsv_hist(xa)}
    side = int(round(math.sqrt(blocks.shape[1])))
    words = nearest_center(blocks.reshape(-1, blocks.shape[2]), C_HOG).view(B, side, side)
    feats[HOG_BOVW] = l2n(grid_hist(words, C_HOG.shape[0]).sqrt()).half().float()
    items, sift_info = [], []
    for k in range(B):
        kps, d, xy = sift_keypoints(grays[k])
        items.append({"kp": (d, xy)})
        kw = (nearest_center(torch.from_numpy(d).to(DEVICE), C_SIFT).cpu().numpy()
              if len(d) else np.zeros(0, np.int64))
        sift_info.append({"gray": grays[k], "kps": kps, "xy": xy, "words": kw})
    feats[SIFT_NAME] = encode_sift_batch(items, C_SIFT, C_SIFT.shape[0]).to(DEVICE).half().float()
    extras = {"cells": cells.cpu(), "lbp_codes": codes.cpu(), "hog_words": words.cpu(), "sift": sift_info}
    return feats, extras


def make_bundle(items, check=True):
    """Images (same resize as extraction) + recomputed features for a list of items."""
    imgs = np.stack([np.asarray(Image.open(it["path"]).convert("RGB").resize((IMG_LOAD, IMG_LOAD), Image.BILINEAR))
                     for it in items])
    grays = [np.asarray(Image.open(it["path"]).convert("L").resize((SIFT_IMG, SIFT_IMG), Image.BILINEAR))
             for it in items]
    feats, extras = compute_features(imgs, grays)
    ds = [k for k, it in enumerate(items) if it["idx"] >= 0]
    if check and ds:
        for name in ["hog", "lbp", "hsv", HOG_BOVW, SIFT_NAME]:
            ref = FEATS[name][[items[k]["idx"] for k in ds]].astype(np.float32).reshape(len(ds), -1)
            diff = np.abs(ref - feats[name][ds].cpu().numpy()).max()
            print(f"  recomputed vs extracted {name:16s} max |diff| = {diff:.2e}")
    return {"items": items, "imgs": imgs, "grays": grays, "feats": feats, "extras": extras}


def predict_bundle(V, keys=None):
    """Predictions and class scores of the trained pipelines on the bundle's images."""
    out = {}
    test_pos = {int(i): k for k, i in enumerate(TEST_IDX)}
    ds = [k for k, it in enumerate(V["items"]) if it["idx"] in test_pos]
    for cfg_name, cfg in TRACK_A.items():
        X = PREPS[cfg_name].transform(torch.cat([normalize_block(V["feats"][n], how) for n, how in cfg["blocks"]], 1))
        for clf in CLASSIFIERS:
            key = f"{cfg_name}|{clf}"
            if keys is not None and key not in keys:
                continue
            model = MODELS[key]
            pred = model.predict_X(X)
            sc, kind = model.scores(X)
            if ds:
                saved = PREDS[key]["y_pred"][[test_pos[V["items"][k]["idx"]] for k in ds]]
                print(f"  {disp(cfg_name):23s} {clf:8s} agrees with the test predictions on "
                      f"{(pred[ds] == saved).mean():.0%} | confidence = {kind}")
            out[key] = {"pred": np.asarray(pred), "scores": np.asarray(sc), "kind": kind}
    return out


def to_probs(scores, kind):
    s = np.asarray(scores, dtype=np.float64)
    if kind == "margin":
        e = np.exp(s - s.max())
        return e / e.sum()
    return s / max(s.sum(), 1e-12)


V_DEMO = make_bundle(DEMO_ITEMS)
P_DEMO = predict_bundle(V_DEMO)

## 11.1 Feature panels

In [ ]:
HSV_BIN_RGB = mcolors.hsv_to_rgb(np.array([[(hb + .5) / 16, (sb + .5) / 4, (vb + .5) / 4]
                                           for hb in range(16) for sb in range(4) for vb in range(4)]))
HOG_WORD_CMAP = mcolors.ListedColormap(np.random.default_rng(0).random((C_HOG.shape[0], 3)) * .85 + .15)
SIFT_WORD_RGB = np.random.default_rng(1).random((C_SIFT.shape[0], 3)) * .85 + .15


def _clean(ax):
    ax.set_xticks([]); ax.set_yticks([])


def panel_hog(ax, V, k):
    c = V["extras"]["cells"][k].numpy()
    nb, Hc, Wc = c.shape
    cs = IMG_SIZE_A / Hc
    vmax = c.max() + 1e-8
    segs, cols = [], []
    for i in range(Hc):
        for j in range(Wc):
            cx, cy = (j + .5) * cs, (i + .5) * cs
            for b in range(nb):
                th = (b + .5) * math.pi / nb + math.pi / 2       # edge is normal to the gradient
                dx, dy = math.cos(th) * cs / 2, math.sin(th) * cs / 2
                segs.append([(cx - dx, cy - dy), (cx + dx, cy + dy)])
                cols.append((1, 1, 1, min(1.0, 1.5 * c[b, i, j] / vmax)))
    ax.add_collection(LineCollection(segs, colors=cols, linewidths=1))
    ax.set_facecolor("black"); ax.set_xlim(0, IMG_SIZE_A); ax.set_ylim(IMG_SIZE_A, 0)
    ax.set_aspect("equal"); _clean(ax)


def panel_hog_energy(ax, V, k):
    e = V["extras"]["cells"][k].sum(0).numpy()
    ax.imshow(V["imgs"][k])
    ax.imshow(e, cmap="inferno", alpha=0.55, extent=(0, IMG_LOAD, IMG_LOAD, 0), interpolation="bilinear")
    _clean(ax); ax.set_xlabel("gradient energy per 8x8 cell", fontsize=7)


def panel_lbp_map(ax, V, k):
    ax.imshow(V["extras"]["lbp_codes"][k], cmap="nipy_spectral", vmin=0, vmax=58, interpolation="nearest")
    _clean(ax)


def panel_lbp_hist(ax, V, k):
    ax.bar(range(59), V["feats"]["lbp"][k, :59].cpu().numpy(), color="#4C72B0", width=1)
    ax.tick_params(labelsize=6); ax.set_xlabel("pattern (58 = non-uniform)", fontsize=7)


def panel_hsv_hist(ax, V, k):
    ax.bar(range(256), V["feats"]["hsv"][k, :256].cpu().numpy(), color=HSV_BIN_RGB, width=1)
    ax.tick_params(labelsize=6); ax.set_xlabel("hue x sat x val bin", fontsize=7)


def panel_hog_words(ax, V, k):
    w = V["extras"]["hog_words"][k]
    ax.imshow(w, cmap=HOG_WORD_CMAP, vmin=0, vmax=C_HOG.shape[0] - 1, interpolation="nearest")
    _clean(ax); ax.set_xlabel(f"{len(np.unique(w))} distinct words / {w.numel()} blocks", fontsize=7)


def panel_hog_word_hist(ax, V, k):
    K = C_HOG.shape[0]
    ax.bar(range(K), V["feats"][HOG_BOVW][k, :K].cpu().numpy(), color="#55A868", width=1)
    ax.tick_params(labelsize=6); ax.set_xlabel("HOG visual word (Hellinger)", fontsize=7)


def panel_sift_kp(ax, V, k):
    info = V["extras"]["sift"][k]
    ax.imshow(cv2.drawKeypoints(info["gray"], info["kps"], None, flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS))
    _clean(ax); ax.set_xlabel(f"{len(info['kps'])} keypoints (size = scale)", fontsize=7)


def panel_sift_words(ax, V, k):
    info = V["extras"]["sift"][k]
    ax.imshow(info["gray"], cmap="gray")
    if len(info["xy"]):
        ax.scatter(info["xy"][:, 0], info["xy"][:, 1], s=9, c=SIFT_WORD_RGB[info["words"]],
                   edgecolors="black", linewidths=0.3)
    ax.axhline(SIFT_IMG / 2, color="white", lw=0.6, ls="--")
    ax.axvline(SIFT_IMG / 2, color="white", lw=0.6, ls="--")
    _clean(ax)
    ax.set_xlabel(f"{len(np.unique(info['words']))} distinct words; dashes = 2x2 pyramid", fontsize=7)


def panel_sift_word_hist(ax, V, k):
    K = C_SIFT.shape[0]
    ax.bar(range(K), V["feats"][SIFT_NAME][k, :K].cpu().numpy(), color="#DD8452", width=1)
    ax.tick_params(labelsize=6); ax.set_xlabel("SIFT visual word (Hellinger)", fontsize=7)


PANELS = {"hog": ("HOG cells", panel_hog), "hog_energy": ("HOG gradient energy", panel_hog_energy),
          "lbp_map": ("LBP code map", panel_lbp_map), "lbp_hist": ("LBP histogram", panel_lbp_hist),
          "hsv_hist": ("HSV histogram", panel_hsv_hist), "hog_words": ("HOG visual words", panel_hog_words),
          "hog_word_hist": ("HOG word histogram", panel_hog_word_hist),
          "sift_kp": ("SIFT keypoints", panel_sift_kp),
          "sift_words": ("keypoints coloured by SIFT word", panel_sift_words),
          "sift_word_hist": ("SIFT word histogram", panel_sift_word_hist)}

# Every Track A feature once: (panel, column header).
FEATURE_COLUMNS = [("hog", "HOG cells"), ("lbp_map", "LBP map"), ("lbp_hist", "LBP hist."),
                   ("hsv_hist", "HSV hist."), ("hog_words", "HOG words"), ("hog_word_hist", "HOG-word hist."),
                   ("sift_words", "SIFT words"), ("sift_word_hist", "SIFT-word hist.")]
FEATURE_WIDTH = 1.2        # width factor of the feature figure (1.0 = square panels)
PRED_PANEL_W = 1.6         # inches per configuration column in the prediction figure


def short_label(it):
    if it["true"] < 0:
        return f"{it.get('short', 'test.png')}\n(no label)"
    return f"#{it['idx']}\n{CLASS_NAMES[it['true']]}"


def compact(ax):
    """Strip a panel down for the dense grid: no ticks, no axis label."""
    ax.set_xticks([]); ax.set_yticks([]); ax.set_xlabel(""); ax.set_ylabel("")
    for s in ("top", "right", "left"):
        ax.spines[s].set_visible(False)


def fig_features(V, columns=FEATURE_COLUMNS, title="Classical ML features on test images",
                 cell=0.92, keep_labels=False):
    """Rows = images; first column the image, then one column per feature panel.
    cell = panel height in inches; keep_labels keeps each panel's axis label (larger figures)."""
    n, nc = len(V["items"]), len(columns) + 1
    fig, axes = plt.subplots(n, nc, figsize=(FEATURE_WIDTH * cell * nc, cell * n + 0.75), squeeze=False)
    for r in range(n):
        ax = axes[r, 0]
        ax.imshow(V["imgs"][r]); ax.set_xticks([]); ax.set_yticks([])
        for sp in ax.spines.values():
            sp.set_visible(False)
        ax.set_ylabel(short_label(V["items"][r]), fontsize=6.5, rotation=0, ha="right", va="center", labelpad=3)
        for j, (name, header) in enumerate(columns, 1):
            ax = axes[r, j]
            PANELS[name][1](ax, V, r)
            if not keep_labels:
                compact(ax)
            if r == 0:
                ax.set_title(header, fontsize=6.5 if not keep_labels else 8, pad=3)
    axes[0, 0].set_title("image", fontsize=6.5 if not keep_labels else 8, pad=3)
    if keep_labels:
        fig.tight_layout(rect=(0, 0, 1, 1 - 0.45 / (cell * n + 0.75)))
    else:
        fig.subplots_adjust(left=0.07, right=0.995, top=1 - 0.55 / (cell * n + 0.75), bottom=0.02,
                            wspace=0.06, hspace=0.06)
    fig.suptitle(title, fontsize=9, fontweight="bold", y=1 - 0.05 / (cell * n + 0.75))
    return fig

## 11.2 Features (every feature once)
HOG: one small star per 8x8 cell (each line an edge direction, brighter = stronger). LBP: texture
code map and its histogram. HSV: colour histogram (bars drawn in the colour they count). HOG words:
every HOG block replaced by its nearest visual word (colour = word id). SIFT: DoG keypoints coloured
by their visual word (dashes = the 2x2 spatial pyramid). Histograms show the whole-image level.

In [ ]:
show_figure(fig_features(V_DEMO))

## 11.3 Predictions of the best classifier of each configuration
Rows = images, columns = configurations; each panel is the class distribution of that configuration's
best classifier (chosen on validation macro-F1), log axis floored at 1e-3. True class shaded grey,
predicted bar outlined, ✓ / ✗ = correct / wrong. For an SVM the bars are the softmax of its decision
values, not a probability.

In [ ]:
def fig_predictions_best(V, P, floor=1e-3):
    items, n = V["items"], len(V["items"])
    cfgs = [c for c in TRACK_A if c in BEST.index and f"{c}|{BEST.loc[c, 'classifier']}" in P]
    H = 0.8 * n + 1.25
    fig, axes = plt.subplots(n, len(cfgs) + 2, figsize=(0.8 + PRED_PANEL_W * len(cfgs), H), squeeze=False,
                             gridspec_kw={"width_ratios": [0.55, 0.22] + [1] * len(cfgs)})
    for ax in axes[:, 1]:                                     # spacer for the y-tick labels
        ax.axis("off")
    axes = np.concatenate([axes[:, :1], axes[:, 2:]], axis=1)
    x = np.arange(N_CLASSES)
    for r, it in enumerate(items):
        ax = axes[r, 0]
        ax.imshow(V["imgs"][r]); ax.set_xticks([]); ax.set_yticks([])
        for sp in ax.spines.values():
            sp.set_visible(False)
        ax.set_ylabel(short_label(it), fontsize=6.5, rotation=0, ha="right", va="center", labelpad=3)
        for j, cfg_name in enumerate(cfgs, 1):
            clf = BEST.loc[cfg_name, "classifier"]
            res_ = P[f"{cfg_name}|{clf}"]
            ax = axes[r, j]
            if it["true"] >= 0:
                ax.axvspan(it["true"] - 0.5, it["true"] + 0.5, color="#9E9E9E", alpha=0.25, lw=0)
            probs = np.clip(to_probs(res_["scores"][r], res_["kind"]), floor, 1)
            p = int(res_["pred"][r])
            bars = ax.bar(x, probs - floor, 0.75, bottom=floor, color="#55A868", alpha=0.9, edgecolor="white", lw=0)
            bars[p].set_edgecolor("black"); bars[p].set_linewidth(0.7)
            if it["true"] >= 0:
                ok = p == it["true"]
                ax.text(0.97, 0.95, "✓" if ok else "✗", transform=ax.transAxes, ha="right", va="top",
                        fontsize=6.5, fontweight="bold", color="#1a7f37" if ok else "#c62828")
            ax.set_yscale("log"); ax.set_ylim(floor, 3); ax.set_xlim(-0.55, N_CLASSES - 0.45)
            ax.minorticks_off(); ax.grid(axis="y", lw=0.4, alpha=0.4)
            ax.set_yticks([1e-3, 1e-2, 1e-1, 1])
            ax.set_yticklabels(["$10^{-3}$", "$10^{-2}$", "$10^{-1}$", "1"] if j == 1 else [], fontsize=5.5)
            ax.set_xticks(x)
            ax.set_xticklabels(CLASS_NAMES if r == n - 1 else [], rotation=90, fontsize=5.5)
            ax.tick_params(length=2, pad=1)
            if r == 0:
                ax.set_title(f"{disp(cfg_name)}\n{CLF_SHORT[clf]} (test F1 {BEST.loc[cfg_name, 'test_f1'] * 100:.1f}%)",
                             fontsize=6, fontweight="bold", pad=3)
    handles = [plt.Rectangle((0, 0), 1, 1, color="#9E9E9E", alpha=0.25, lw=0)]
    labels = ["true class"]
    if any(P[f"{c}|{BEST.loc[c, 'classifier']}"]["kind"] != "prob" for c in cfgs):
        handles.append(plt.Rectangle((0, 0), 1, 1, color="#55A868", ec="white", lw=0))
        labels.append("SVM: softmax of decision values")
    fig.legend(handles, labels, loc="upper center", ncol=len(handles), fontsize=6,
               bbox_to_anchor=(0.53, 1 - 0.32 / H), handlelength=1.4, columnspacing=1.2)
    fig.subplots_adjust(left=0.07, right=0.995, top=1 - 0.78 / H, bottom=0.55 / H, wspace=0.08, hspace=0.12)
    fig.suptitle("Class probabilities (log scale) of the best classifier of each configuration\n"
                 "✓ / ✗ = correct / wrong; outlined bar = prediction", fontsize=8, fontweight="bold", y=1.0)
    return fig


show_figure(fig_predictions_best(V_DEMO, P_DEMO))

## 11.4 Inference FLOPs and model size (20 test images)
For each of the 15 models: FLOPs of one prediction (features + preprocessing + classifier), averaged
over `COST_N` fixed test images, and the model size.

* PyTorch parts (HOG, LBP, HSV, visual-word assignment and pyramid histograms, PCA + standardization,
  logistic regression) are counted op by op: matmul with PyTorch's formula (1 multiply-add = 2 FLOPs),
  1 FLOP per output element for the other arithmetic ops.
* Parts outside PyTorch are **estimated** ("est." columns): SIFT (Gaussian pyramid, DoG, extremum test,
  orientation + descriptor per keypoint), RBF-SVM (one kernel value per support vector + the weighted
  sums), XGBoost (one comparison per tree level actually visited, from `pred_leaf`).
* Image decoding and resizing (PIL) are not counted.
* Size: classifier parameters, preprocessing (PCA + standardization) and codebook parameters, and the
  size of the serialized model + preprocessing.

In [ ]:
from torch.utils._python_dispatch import TorchDispatchMode
from torch.utils.flop_counter import FlopCounterMode, flop_registry

# aten ops that only move, copy, view or create data: not counted as FLOPs
_NO_FLOP_OPS = {
    "view", "_unsafe_view", "reshape", "_reshape_alias", "expand", "expand_as", "view_as", "permute",
    "transpose", "t", "clone", "copy_", "_to_copy", "to", "detach", "alias", "lift_fresh", "lift_fresh_copy",
    "slice", "select", "index", "index_select", "gather", "scatter", "index_put_", "masked_fill", "masked_fill_",
    "cat", "stack", "split", "split_with_sizes", "unbind", "chunk", "narrow", "unfold", "flip", "roll", "repeat",
    "squeeze", "squeeze_", "unsqueeze", "unsqueeze_", "contiguous", "as_strided", "diagonal", "tril", "triu",
    "empty", "empty_like", "empty_strided", "zeros", "zeros_like", "ones", "ones_like", "full", "full_like",
    "fill_", "zero_", "new_zeros", "new_ones", "new_empty", "new_full", "arange", "scalar_tensor", "randperm",
    "_local_scalar_dense", "item", "set_", "resize_", "constant_pad_nd", "pad", "embedding", "where",
    "_assert_async", "_assert_tensor_metadata", "is_nonzero", "equal"}


class _OtherOpsCounter(TorchDispatchMode):
    """1 FLOP per output element for every aten op that is not a matmul/conv/attention
    (those are counted by FlopCounterMode) and not a pure data-movement op."""
    def __init__(self):
        super().__init__()
        self.n = 0

    def __torch_dispatch__(self, func, types, args=(), kwargs=None):
        out = func(*args, **(kwargs or {}))
        pk = func.overloadpacket
        if pk not in flop_registry and pk.__name__.rstrip("_") not in _NO_FLOP_OPS \
                and pk.__name__ not in _NO_FLOP_OPS:
            for t in (out if isinstance(out, (tuple, list)) else (out,)):
                if isinstance(t, torch.Tensor):
                    self.n += t.numel()
        return out


def count_flops(fn):
    """Run fn() once; returns (matmul/conv/attention FLOPs, other-op FLOPs, fn's result).
    Work done outside PyTorch (PIL, NumPy, OpenCV, scikit-learn, XGBoost) is not seen."""
    other = _OtherOpsCounter()
    with FlopCounterMode(display=False) as fc:
        with other:
            out = fn()
    return float(fc.get_total_flops()), float(other.n), out


def pick_cost_images(test_idx, n, seed):
    rng = np.random.default_rng(seed)
    return np.sort(rng.choice(np.asarray(test_idx), n, replace=False)).astype(np.int64)


COST_IDX = pick_cost_images(TEST_IDX, COST_N, SEED + 20)


def load_pair(path):
    """The two inputs of the pipelines, resized exactly as in extraction."""
    rgb = np.asarray(Image.open(path).convert("RGB").resize((IMG_LOAD, IMG_LOAD), Image.BILINEAR))
    gray = np.asarray(Image.open(path).convert("L").resize((SIFT_IMG, SIFT_IMG), Image.BILINEAR))
    return rgb, gray


def torch_blocks(names, rgb, sift):
    """The torch part of the feature blocks in `names` for one image (SIFT descriptors given)."""
    x = torch.from_numpy(np.ascontiguousarray(rgb[None])).to(DEVICE).permute(0, 3, 1, 2).float() / 255
    xa = F.interpolate(x, size=(IMG_SIZE_A, IMG_SIZE_A), mode="bilinear", antialias=True,
                       align_corners=False).clamp(0, 1)
    out = {}
    if "hog" in names or HOG_BOVW in names:
        blocks = hog_blocks(xa).half().float()
        if "hog" in names:
            out["hog"] = blocks.reshape(1, -1)
        if HOG_BOVW in names:
            side = int(round(math.sqrt(blocks.shape[1])))
            words = nearest_center(blocks.reshape(-1, blocks.shape[2]), C_HOG).view(1, side, side)
            out[HOG_BOVW] = l2n(grid_hist(words, C_HOG.shape[0]).sqrt()).half().float()
    if "lbp" in names:
        out["lbp"] = lbp_hist(rgb_to_gray(xa))
    if "hsv" in names:
        out["hsv"] = hsv_hist(xa)
    if SIFT_NAME in names:
        out[SIFT_NAME] = encode_sift_batch([{"kp": sift}], C_SIFT, C_SIFT.shape[0]).to(DEVICE).half().float()
    return out


def sift_flops_est(n_kp, size=SIFT_IMG, layers=3, sigma=1.6):
    """Rough FLOP estimate of OpenCV SIFT detect + compute on a size x size image (Lowe, 2004)."""
    px = (2 * size) ** 2 * 4 / 3                         # 2x upsampled base image, all octaves
    taps = 2 * math.ceil(3 * sigma) + 1                    # average separable Gaussian kernel
    blur = px * (layers + 3) * 2 * 2 * taps               # 2 passes, multiply + add per tap
    dog = px * (layers + 2)
    extrema = px * layers * 26
    per_kp = 625 * 10 + 600 * 25 + 128 * 3                 # orientation histogram + descriptor + RootSIFT
    return blur + dog + extrema + n_kp * per_kp


def svm_sv_groups(m):
    """Support-vector counts: [n] for one shared set (scikit-learn), else one count per binary SVC (cuML)."""
    sv = None
    try:
        sv = np.asarray(m.support_vectors_)
    except Exception:
        pass
    if sv is not None and sv.ndim == 2 and len(sv):
        return [sv.shape[0]], True
    ests = []
    for path in ("multiclass_svc.multiclass_estimator.estimators_", "estimators_"):
        obj = m
        try:
            for a in path.split("."):
                obj = getattr(obj, a)
            ests = list(obj); break
        except Exception:
            continue
    return [np.asarray(e.support_vectors_).shape[0] for e in ests], False


def svm_flops_est(svm, d):
    groups, shared = svm_sv_groups(svm.m)
    if not groups:
        return np.nan, np.nan
    if shared:                                            # one-vs-one over one shared set of SVs
        n = groups[0]
        return n * (3 * d + 2) + 2 * n * (N_CLASSES - 1), n * (d + 1)
    return sum(n * (3 * d + 2) + 2 * n for n in groups), sum(n * (d + 1) for n in groups)


_XGB_DEPTH = {}


def xgb_flops_est(model, X, key):
    """Exact number of tree levels visited (pred_leaf) + 1 add per tree + softmax."""
    booster = model.m.get_booster()
    if key not in _XGB_DEPTH:
        df = booster.trees_to_dataframe()
        depth = {}
        for t, g in df.sort_values(["Tree", "Node"]).groupby("Tree"):
            d = {0: 0}
            for node, yes, no in zip(g["Node"], g["Yes"], g["No"]):
                if isinstance(yes, str):
                    for c in (yes, no):
                        d[int(c.split("-")[1])] = d[node] + 1
            depth[t] = d
        _XGB_DEPTH[key] = (depth, len(df))
    depth, n_nodes = _XGB_DEPTH[key]
    try:
        it_range = (0, int(model.m.best_iteration) + 1)
    except Exception:
        it_range = (0, 0)
    leaves = booster.predict(xgb.DMatrix(X), pred_leaf=True, iteration_range=it_range).reshape(-1)
    comps = sum(depth[t][int(l)] for t, l in enumerate(leaves))
    return comps + len(leaves) + 3 * N_CLASSES, n_nodes


def serialized_mb(cfg_name, clf):
    """Size of the model + preprocessing when saved to disk (written to a temporary folder)."""
    with tempfile.TemporaryDirectory() as d:
        p = os.path.join(d, f"model.{CLF_EXT[clf]}")
        MODELS[f"{cfg_name}|{clf}"].save(p)
        q = os.path.join(d, "prep.pt")
        torch.save([None if p_ is None else {k: (v.cpu() if torch.is_tensor(v) else v) for k, v in p_.items()}
                    for p_ in PREPS[cfg_name].params], q)
        return (os.path.getsize(p) + os.path.getsize(q)) / 2 ** 20


@torch.no_grad()
def ml_flops(cfg_name, clf):
    cfg = TRACK_A[cfg_name]
    names = [n for n, _ in cfg["blocks"]]
    prep, model = PREPS[cfg_name], MODELS[f"{cfg_name}|{clf}"]
    rows, clf_params = [], np.nan
    for i in COST_IDX:
        rgb, gray = load_pair(PATHS[i])
        sift, n_kp = (None, None), 0
        if SIFT_NAME in names:
            kps, d, xy = sift_keypoints(gray)
            sift, n_kp = (d, xy), len(kps)
        f_mm, f_ot, feats = count_flops(lambda: torch_blocks(names, rgb, sift))
        p_mm, p_ot, X = count_flops(lambda: prep.transform(
            torch.cat([normalize_block(feats[n], how) for n, how in cfg["blocks"]], 1)))
        if clf == "logreg":
            c_mm, c_ot, _ = count_flops(lambda: model.predict_X(X))
            c_flops, clf_params = c_mm + c_ot, model.W.numel() + model.b.numel()
        elif clf == "svm_rbf":
            c_flops, clf_params = svm_flops_est(model, X.shape[1])
        else:
            c_flops, clf_params = xgb_flops_est(model, to_np(X), f"{cfg_name}|{clf}")
        rows.append([f_mm + f_ot, sift_flops_est(n_kp) if SIFT_NAME in names else 0.0, p_mm + p_ot, c_flops])
    cb_params = sum(CODEBOOKS[n].numel() for n in names if n in CODEBOOKS)
    return np.array(rows, dtype=np.float64), clf_params, prep.n_params(), cb_params, serialized_mb(cfg_name, clf)


def build_ml_cost():
    rows = []
    for c, k in RUNS:
        f, clf_p, prep_p, cb_p, mb = ml_flops(c, k)
        tot = f.sum(1)
        rows.append({"pipeline": disp(c), "config": c, "classifier": k, "n_images": len(tot),
                     "MFLOPs_mean": tot.mean() / 1e6, "MFLOPs_std": tot.std() / 1e6,
                     "MFLOPs_min": tot.min() / 1e6, "MFLOPs_max": tot.max() / 1e6,
                     "MFLOPs_features_torch": f[:, 0].mean() / 1e6, "MFLOPs_sift_opencv_est": f[:, 1].mean() / 1e6,
                     "MFLOPs_pca_std": f[:, 2].mean() / 1e6, "MFLOPs_classifier": f[:, 3].mean() / 1e6,
                     "classifier_flops_estimated": k != "logreg",
                     "classifier_params": clf_p, "prep_params": prep_p, "codebook_params": cb_p,
                     "total_params_M": (np.nan_to_num(clf_p) + prep_p + cb_p) / 1e6, "serialized_MB": mb})
    return pd.DataFrame(rows)


_ = show_table(build_ml_cost())